# 10 — Operational Prioritization

## Objective

This notebook loads the exact fitted standard-Python model bundle from Notebook 08, scores the operational evaluation period, assigns risk levels using predicted probabilities, and evaluates capacity-based prioritization. It does not assume a specific winning algorithm.


In [ ]:
from __future__ import annotations

import importlib.util
import json
import os
from pathlib import Path

bootstrap = Path.cwd() / "notebooks" / "import_path.py"
if not bootstrap.is_file():
    bootstrap = Path.cwd() / "import_path.py"
spec = importlib.util.spec_from_file_location("import_path", bootstrap)
ip = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ip)

import joblib
import numpy as np
import pandas as pd

from pyspark.sql import functions as F
from config import project_config as cfg

RANDOM_SEED = cfg.RANDOM_SEED
TARGET_COLUMN = cfg.TARGET_COLUMN

def filesystem_path(path_value):
    path_value = str(path_value)
    return "/dbfs/" + path_value[6:] if path_value.startswith("dbfs:/") else path_value

bundle_path = os.path.join(filesystem_path(cfg.SELECTED_MODEL_PATH), "model_bundle.joblib")
bundle = joblib.load(bundle_path)
pipeline = bundle["pipeline"]
model_name = bundle["model_name"]
threshold = float(bundle["decision_threshold"])
manifest = bundle["feature_manifest"]
input_columns = list(manifest["model_input_columns"])

print(f"Operational scoring model: {model_name}")
print(f"Stored classification threshold: {threshold:.4f}")


## Score the Operational Period

The scoring sample retains identifying and scheduling fields needed by the dashboard. Predicted probabilities are produced by the same fitted pipeline evaluated in Notebook 08.


In [ ]:
source_df = spark.table(cfg.MODELING_VALIDATION_HIST_TABLE)
available_columns = set(source_df.columns)

identity_candidates = [
    "FL_DATE", "OP_UNIQUE_CARRIER", "ORIGIN", "DEST",
    "ORIGIN_CITY_NAME", "DEST_CITY_NAME", "CRS_DEP_TIME",
    "MONTH", TARGET_COLUMN,
]
identity_columns = [column for column in identity_candidates if column in available_columns]
selected_columns = list(dict.fromkeys(input_columns + identity_columns))

maximum_rows = int(getattr(cfg, "OPERATIONAL_SCORING_MAX_ROWS", 200_000))
scoring_df = source_df.select(*selected_columns)
if scoring_df.count() > maximum_rows:
    scoring_df = scoring_df.orderBy(F.rand(RANDOM_SEED)).limit(maximum_rows)
scoring_pdf = scoring_df.toPandas()

probabilities = pipeline.predict_proba(scoring_pdf[input_columns])[:, 1]
predictions = (probabilities >= threshold).astype(int)
scoring_pdf["delay_probability"] = probabilities
scoring_pdf["predicted_delay"] = predictions
scoring_pdf["actual_delay"] = scoring_pdf[TARGET_COLUMN].astype(int)

high_threshold = float(getattr(cfg, "HIGH_RISK_THRESHOLD", 0.60))
medium_threshold = float(getattr(cfg, "MEDIUM_RISK_THRESHOLD", 0.35))
scoring_pdf["risk_level"] = np.select(
    [probabilities >= high_threshold, probabilities >= medium_threshold],
    ["High", "Medium"],
    default="Low",
)


In [ ]:
global_importance = spark.table(cfg.SHAP_GLOBAL_IMPORTANCE_TABLE).orderBy(F.desc("MeanAbsSHAP"))
top_row = global_importance.first()
top_driver = top_row["Feature"] if top_row else "Model probability"

def present(column, default=None):
    if column in scoring_pdf.columns:
        return scoring_pdf[column]
    return pd.Series([default] * len(scoring_pdf), index=scoring_pdf.index)

predictions_pdf = pd.DataFrame({
    "flight_label": (
        present("OP_UNIQUE_CARRIER", "Flight").astype(str)
        + " " + np.arange(len(scoring_pdf)).astype(str)
    ),
    "airline_code": present("OP_UNIQUE_CARRIER", "Unknown"),
    "origin_airport": present("ORIGIN", "Unknown"),
    "destination_airport": present("DEST", "Unknown"),
    "scheduled_departure_text": present("CRS_DEP_TIME", "Unknown").astype(str),
    "departure_window": present("FL_DATE", "Unknown").astype(str),
    "delay_probability": scoring_pdf["delay_probability"],
    "risk_level": scoring_pdf["risk_level"],
    "month_number": present("MONTH", 0),
    "shap_main_driver": top_driver,
    "actual_delay": scoring_pdf["actual_delay"],
    "predicted_delay": scoring_pdf["predicted_delay"],
})

predictions_spark = spark.createDataFrame(predictions_pdf)
(
    predictions_spark.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(cfg.PREDICTIONS_TABLE)
)

predictions_path = filesystem_path(cfg.PREDICTIONS_PATH)
os.makedirs(predictions_path, exist_ok=True)
predictions_pdf.to_parquet(os.path.join(predictions_path, "predictions.parquet"), index=False)

display(predictions_spark.orderBy(F.desc("delay_probability")).limit(20))


## Capacity-Based Prioritization

For each capacity level `K`, the highest-risk `K` flights are prioritized. Recall at K measures the share of all observed delays captured by those alerts; Lift at K compares the prioritized delay rate with the overall delay prevalence.


In [ ]:
capacity_options = list(getattr(cfg, "CAPACITY_K_OPTIONS", [100, 500, 1_000, 5_000]))
ordered = predictions_pdf.sort_values("delay_probability", ascending=False).reset_index(drop=True)
overall_rate = max(float(ordered["actual_delay"].mean()), 1e-12)
total_delays = max(int(ordered["actual_delay"].sum()), 1)

evaluation_rows = []
for requested_k in capacity_options:
    k = min(int(requested_k), len(ordered))
    prioritized = ordered.head(k)
    captured = int(prioritized["actual_delay"].sum())
    evaluation_rows.append({
        "capacity_k": int(k),
        "delays_captured": captured,
        "recall_at_k": float(captured / total_delays),
        "precision_at_k": float(prioritized["actual_delay"].mean()),
        "lift_at_k": float(prioritized["actual_delay"].mean() / overall_rate),
        "model_name": model_name,
    })

evaluation_spark = spark.createDataFrame(pd.DataFrame(evaluation_rows))
(
    evaluation_spark.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(cfg.PRIORITIZATION_EVALUATION_TABLE)
)

evaluation_path = filesystem_path(cfg.PRIORITIZATION_EVALUATION_PATH)
os.makedirs(evaluation_path, exist_ok=True)
pd.DataFrame(evaluation_rows).to_parquet(
    os.path.join(evaluation_path, "prioritization_evaluation.parquet"),
    index=False,
)

display(evaluation_spark.orderBy("capacity_k"))


## Operational Handoff

The predictions and prioritization tables retain their established names for Notebook 11. The probability scores now come from the actual dynamically selected Python model and its persisted preprocessing pipeline.
